# SAE Test-Set Reconstruction → SAC Policy Evaluation

**Overview:** Evaluate SAE reconstruction quality on the test set and quantify any performance change when a pretrained SAC policy acts on *reconstructed* vs *original* states.

## Goals
- Show loss on test examples (MSE, sparsity)
- Export reconstructed states for download
- Analyze performance loss of SAC with reconstructed states

In [1]:
import os
import sys
sys.path.append('../')
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
from models.sae import SAE
from scripts.data import DataLoader

Root = Path().resolve().parent

In [2]:
def test(dataset, sae, topk, batch_size=100000):
    size = 100000 - (100000 % batch_size)
    original = np.array(np.zeros((size, 348)))
    reconstructed = np.array(np.zeros((size, 348)))
    for i in range(size // batch_size):
        batch = dataset.sample(batch_size, dataset="Test")
        y = sae.forward(batch, topk=topk)
        original[i*batch_size:(i+1)*batch_size] = batch
        reconstructed[i*batch_size:(i+1)*batch_size] = y.detach().numpy()
        
        # compute loss
        recon_loss = F.mse_loss(y, batch)
        l0_norm = (sae.z != 0).float().sum(dim=1).mean().item()

        # print info
        if i % 1 == 0:
            print(f"step: {i+1}, loss: {recon_loss:.02f}, l0 norm: {l0_norm:.1f}")
        # if i % 200000:
        #     print(batch[0] - y[0])
    
    np.save("original", original)      
    np.save("reconstructed", reconstructed)
    

def load(sae):
    path = Root / f"checkpoints/sae"
    model = torch.load(path / "model.pth")
    sae.load_state_dict(model)

In [3]:
sae = SAE(348, 4, 348, True)
load(sae)
dataset = DataLoader()
test(dataset, sae, topk=True)

step: 1, loss: 0.03, l0 norm: 16.0


# Evaluation of Reconstructed States

| Metric                           |     Value |
|----------------------------------|----------:|
| **Average Expected Return Δ**    | **-7.1050** |
| **Average Return**               | **449.5003** |
| **Average Reconstructed Return** | **456.6086** |

<sub>Δ = difference in expected return (sign per your definition).</sub>